# 06 — LLM Reasoning Layer

**CineMind Phase 1 · Step 6** — Wrap the trained retrieval artifacts (content
vectors from notebook 03, collaborative item vectors from notebook 04) in a
LangChain reasoning layer powered by the Claude API.

### What this notebook builds (LCEL chains)

1. **Query parsing** — turn a free-text request ("something like Inception but
   lighter") into a structured filter (genres, mood, keywords, exclude-list).
2. **Grounded retrieval (RAG)** — embed the parsed query with the same E5 model
   from notebook 03, search the real content vectors, and pass only *real*
   movie titles forward. The LLM never invents a title — it only reasons over
   rows we actually retrieved.
3. **Re-ranking** — ask Claude to pick and order the best 10 of ~30 retrieved
   candidates for the stated query, returned as strict JSON `movie_id`s that
   must be a subset of what was retrieved (no hallucinated ids either).
4. **"Why this?" explanations** — a short, grounded explanation for a single
   recommendation, citing only metadata we actually have.
5. **Cold-start onboarding chat** — a tiny multi-turn dialogue that converts a
   new user's answers into a taste profile, then retrieves seed
   recommendations via content vectors (no collaborative history needed).
6. **Langfuse tracing** — every chain call traced with cost/latency/inputs if
   Langfuse keys are configured; skipped gracefully otherwise.

### Requires
```
artifacts/content_vecs.npy, artifacts/content_movie_ids.npy   (from notebook 03)
artifacts/item_vecs.npy                                        (from notebook 04)
data/items.csv                                                 (from notebook 01)
```
### Needs (only for the LLM cells — retrieval cells work without them)
```
pip install langchain langchain-anthropic langfuse python-dotenv
```
And in a `.env` file at the project root (never commit this file):
```
ANTHROPIC_API_KEY=sk-ant-...
LANGFUSE_PUBLIC_KEY=pk-lf-...   # optional
LANGFUSE_SECRET_KEY=sk-lf-...   # optional
LANGFUSE_HOST=https://cloud.langfuse.com   # optional
```
This notebook follows the same graceful-degradation pattern as notebook 05:
retrieval cells always run; LLM cells detect whether a key + packages are
present and clearly print what to install/set if not, instead of crashing.

## 1 · Setup

In [1]:
import os, sys, json

# Run from project root regardless of where Jupyter was launched
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Make src/ importable
if not any('src' in p for p in sys.path):
    sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import cinemind_utils as cu

# Load .env if python-dotenv is available (keeps API keys out of the notebook)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Provider selection -- first one with a key wins. Anthropic is paid;
# Groq is a free-tier fallback (console.groq.com) so this notebook runs
# for free out of the box. Set ANTHROPIC_API_KEY later and it takes over
# automatically -- no other code changes needed.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

LLM_PROVIDER = None   # "anthropic" | "groq" | None
haiku = None            # small/cheap model -- parsing, explaining, onboarding
sonnet = None           # larger model -- re-ranking (needs more judgement)

if os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from langchain_anthropic import ChatAnthropic
        haiku = ChatAnthropic(model="claude-3-5-haiku-latest", temperature=0)
        sonnet = ChatAnthropic(model="claude-sonnet-4-5", temperature=0.3)
        LLM_PROVIDER = "anthropic"
    except ImportError:
        pass

if LLM_PROVIDER is None and os.environ.get("GROQ_API_KEY"):
    try:
        from langchain_groq import ChatGroq
        haiku = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
        sonnet = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
        LLM_PROVIDER = "groq"
    except ImportError:
        pass

LLM_AVAILABLE = LLM_PROVIDER is not None

print("cwd:", os.getcwd())
print("LLM provider:", LLM_PROVIDER or "none")
print("LLM chains will run:", LLM_AVAILABLE)
if not LLM_AVAILABLE:
    print("Retrieval cells below still work without an API key.")
    print("To enable the LLM chains for free: pip install langchain-groq,")
    print("get a key at console.groq.com, and set GROQ_API_KEY (in .env).")
    print("Or set ANTHROPIC_API_KEY if you would rather use Claude (paid).")

cwd: C:\Users\shaur\Desktop\My codes shaurya\Content Recommendation system\cinemind _phase1\cinemind_phase1
LLM provider: groq
LLM chains will run: True


## 2 · Load artifacts

In [2]:
for f in ["artifacts/content_vecs.npy", "artifacts/content_movie_ids.npy",
          "artifacts/item_vecs.npy", "data/items.csv"]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing {f}. Run notebooks 01, 03, 04 first.")

items = pd.read_csv("data/items.csv")
content_vecs = np.load("artifacts/content_vecs.npy").astype(np.float32)
content_ids = np.load("artifacts/content_movie_ids.npy")
item_vecs = np.load("artifacts/item_vecs.npy").astype(np.float32)

# Normalise content vectors once so dot product == cosine similarity
content_vecs = content_vecs / np.linalg.norm(content_vecs, axis=1, keepdims=True)

title_of = dict(zip(items.movie_id, items.title))
genres_of = dict(zip(items.movie_id, items.genres))
content_row_of_movie = dict(zip(content_ids.tolist(), range(len(content_ids))))

# Optional: poster/overview/cast from step 07 (OMDb enrichment). ml-100k
# ships no links.csv, so movies are matched to OMDb by title+year, not a
# join -- see src/07_omdb_enrichment.py. Safe to skip; downstream code
# treats a missing/absent movie_meta.csv as "no enrichment yet".
enrich_of = {}
if os.path.exists("artifacts/movie_meta.csv"):
    meta_df = pd.read_csv("artifacts/movie_meta.csv")
    enrich_of = {
        int(r.movie_id): {
            "poster_url": r.poster_url if pd.notna(r.poster_url) else None,
            "overview": r.overview if pd.notna(r.overview) else None,
            "cast": r.cast if pd.notna(r.cast) else None,
        }
        for r in meta_df.itertuples()
    }

print(f"Movies: {len(items):,}  Content vectors: {content_vecs.shape}  "
      f"Collab vectors: {item_vecs.shape}")
print(f"OMDb enrichment: {len(enrich_of):,} movies matched")


Movies: 1,682  Content vectors: (1682, 384)  Collab vectors: (1683, 64)
OMDb enrichment: 0 movies matched


## 3 · Grounded retrieval (RAG)

Embed the free-text query with the *same* E5 model used to build the content
vectors in notebook 03, then do plain cosine search. This step needs no LLM —
it only needs `sentence-transformers`, and it is what prevents hallucination:
every candidate handed to the LLM later is a real row from `items.csv`.

In [3]:
_e5_model = None

def get_e5():
    """Lazy-load the E5 model once (same one as notebook 03)."""
    global _e5_model
    if _e5_model is None:
        from sentence_transformers import SentenceTransformer
        _e5_model = SentenceTransformer("intfloat/e5-small-v2")
    return _e5_model


def retrieve_candidates(query, k=30):
    """Free-text query -> top-k real movies by content similarity.

    Returns a list of dicts: {movie_id, title, genres, score}. This is the
    ONLY thing later chains are allowed to reason over, so nothing downstream
    can invent a movie that isn't in the catalogue.
    """
    model = get_e5()
    # E5 expects a "query:" prefix for search queries (vs "passage:" for docs)
    q_vec = model.encode(f"query: {query}", normalize_embeddings=True)
    sims = content_vecs @ q_vec
    order = np.argsort(-sims)[:k]
    out = []
    for row in order:
        mid = int(content_ids[row])
        extra = enrich_of.get(mid, {})
        out.append({
            "movie_id": mid,
            "title": title_of.get(mid, "Unknown"),
            "genres": genres_of.get(mid, "[]"),
            "score": float(sims[row]),
            "poster_url": extra.get("poster_url"),
            "overview": extra.get("overview"),
            "cast": extra.get("cast"),
        })
    return out


# Quick sanity check (no LLM needed)
demo_candidates = retrieve_candidates("a lighthearted animated adventure for kids", k=5)
for c in demo_candidates:
    print(f"  {c['score']:.3f}  {c['title']:40s} {c['genres']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  0.819  Adventures of Pinocchio, The (1996)      ['Adventure', "Children's"]
  0.817  Flintstones, The (1994)                  ["Children's", 'Comedy']
  0.815  Secret Adventures of Tom Thumb, The (1993) ['Adventure', "Children's"]
  0.814  Amazing Panda Adventure, The (1995)      ['Adventure', "Children's"]
  0.814  Pinocchio (1940)                         ['Animation', "Children's"]


## 4 · LangChain chains (LCEL)

Three chains, each a small `prompt | llm | parser` pipeline:

- **parse_query_chain** — free text -> structured search intent (JSON)
- **rerank_chain** — retrieved candidates + intent -> ordered top 10 (JSON,
  ids constrained to the candidate set)
- **explain_chain** — one movie + user context -> a short grounded explanation

Haiku is used for parsing/explaining (cheap, fast); Sonnet for re-ranking
(needs more judgement across candidates), matching the model split in
`CLAUDE.md`.

In [4]:
import json
import re
from langchain_core.runnables import RunnableLambda


def parse_json_loose(message):
    """Extract a JSON value from an LLM response, tolerating the extra
    prose / markdown fences / inline '#' comments that free-tier models
    (Groq's Llama in particular) sometimes add despite being told
    'JSON only' -- Claude follows that instruction strictly, but we can't
    assume every provider will.
    """
    text = message.content if hasattr(message, "content") else str(message)
    text = re.sub(r"#.*", "", text)
    match = re.search(r"(\[.*\]|\{.*\})", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in LLM output: {text[:200]}")
    return json.loads(match.group(1))


json_parser = RunnableLambda(parse_json_loose)

if LLM_AVAILABLE:
    # ---- 4a. Query parsing --------------------------------------------
    parse_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You turn a free-text movie request into a JSON search intent. "
         "Return ONLY JSON with keys: search_text (a short phrase to embed "
         "for similarity search), mood (one or two words), exclude_genres "
         "(list, may be empty). No prose, no markdown fences."),
        ("human", "{query}"),
    ])
    parse_query_chain = parse_prompt | haiku | json_parser

    # ---- 4b. Re-ranking -------------------------------------------------
    rerank_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You rank movies for a user request. You may ONLY choose from the "
         "candidate list given to you -- never invent a movie_id or title. "
         "Respond with NOTHING but a JSON array of up to 10 movie_id integers, "
         "best match first -- no explanation, no comments, no markdown fences, "
         "no text before or after the array."),
        ("human",
         "User request: {query}" + chr(10) + chr(10) + "Candidates (movie_id | title | genres):" + chr(10) +
         "{candidates}"),
    ])
    rerank_chain = rerank_prompt | sonnet | json_parser

    # ---- 4c. "Why this?" explanation -----------------------------------
    explain_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Explain in 1-2 sentences why this movie fits the user's request. "
         "Ground your answer in the genres given, and in the plot summary "
         "when one is provided -- do not invent plot details, cast, or "
         "facts beyond what's given."),
        ("human",
         "User request: {query}" + chr(10) + "Movie: {title} | Genres: {genres}" + chr(10) +
         "Plot: {overview}"),
    ])
    explain_chain = explain_prompt | haiku | StrOutputParser()

    print(f"Chains ready ({LLM_PROVIDER}): parse_query_chain, rerank_chain, explain_chain")
else:
    print("Skipped -- LLM_AVAILABLE is False (see setup cell above).")

Chains ready (groq): parse_query_chain, rerank_chain, explain_chain


## 5 · End-to-end conversational search

`query -> parse -> retrieve (grounded) -> rerank -> explain top pick`

This is the same pattern `backend/llm_chains.py` will expose over `POST /chat`
in Phase 2 — this notebook is where it gets proven out.

In [5]:
def conversational_search(query, k_candidates=30, k_final=10):
    if not LLM_AVAILABLE:
        print("LLM_AVAILABLE is False -- showing retrieval-only results.")
        return retrieve_candidates(query, k=k_final)

    intent = parse_query_chain.invoke({"query": query})
    search_text = intent.get("search_text", query)

    candidates = retrieve_candidates(search_text, k=k_candidates)
    exclude = set(g.lower() for g in intent.get("exclude_genres", []))
    if exclude:
        candidates = [c for c in candidates
                      if not exclude & set(g.lower() for g in eval(c["genres"]))]

    candidate_lines = "\n".join(
        f"{c['movie_id']} | {c['title']} | {c['genres']}" for c in candidates
    )
    ranked_ids = rerank_chain.invoke({"query": query, "candidates": candidate_lines})

    by_id = {c["movie_id"]: c for c in candidates}
    seen_ids = set()
    results = []
    for i in ranked_ids:
        # The rerank LLM occasionally repeats a movie_id in its output --
        # de-dupe while preserving its ranking order.
        if i in by_id and i not in seen_ids:
            seen_ids.add(i)
            results.append(by_id[i])
    results = results[:k_final]

    if results:
        top = results[0]
        why = explain_chain.invoke({
            "query": query, "title": top["title"], "genres": top["genres"],
            "overview": top.get("overview") or "not available",
        })
        top["why"] = why
    return results


if LLM_AVAILABLE:
    results = conversational_search("something like Inception but not so dark")
    for r in results:
        print(f"  {r['movie_id']:>4}  {r['title']}")
    if results:
        print("\nWhy the top pick:", results[0].get("why"))
else:
    demo = conversational_search("something like Inception but not so dark")
    for r in demo:
        print(f"  {r['movie_id']:>4}  {r['title']:40s} (retrieval-only, no LLM rerank/explain)")

    82  Jurassic Park (1993)
    62  Stargate (1994)
   797  Timecop (1994)
   222  Star Trek: First Contact (1996)
   250  Fifth Element, The (1997)
   380  Star Trek: Generations (1994)
   229  Star Trek III: The Search for Spock (1984)
   230  Star Trek IV: The Voyage Home (1986)
   271  Starship Troopers (1997)
   235  Mars Attacks! (1996)

Why the top pick: Jurassic Park fits the user's request because it combines elements of action and adventure with a sci-fi twist, creating a thrilling experience without being as dark as Inception. The movie's focus on a theme park filled with cloned dinosaurs provides a sense of wonder and excitement, rather than the complex, layered narrative of Inception.


## 6 · Cold-start onboarding chat

A brand-new user has no ratings, so the collaborative two-tower model has
nothing to work with. Instead we ask a couple of onboarding questions, turn
the answers into a taste profile with the LLM, and retrieve seed
recommendations purely from **content** vectors -- solving user cold-start
without needing any interaction history.

In [6]:
onboarding_prompt = None
onboarding_chain = None

if LLM_AVAILABLE:
    onboarding_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "A new user answered onboarding questions about movies they like. "
         "Summarise their taste as a short search phrase (one sentence) "
         "suitable for a semantic movie search. Return ONLY that sentence, "
         "no preamble."),
        ("human", "{answers}"),
    ])
    onboarding_chain = onboarding_prompt | haiku | StrOutputParser()


def onboard_new_user(answers):
    """answers: free text describing favourite movies/genres/mood."""
    if not LLM_AVAILABLE:
        print("LLM_AVAILABLE is False -- using raw answers as the search text.")
        return retrieve_candidates(answers, k=10)
    taste_summary = onboarding_chain.invoke({"answers": answers})
    print("Taste summary:", taste_summary)
    return retrieve_candidates(taste_summary, k=10)


demo_answers = (
    "I loved The Matrix and Toy Story. I like sci-fi and animation, "
    "not really into horror or war movies."
)
seed_recs = onboard_new_user(demo_answers)
for r in seed_recs:
    print(f"  {r['movie_id']:>4}  {r['title']}")

Taste summary:

 Sci-fi animation films.
  1240  Ghost in the Shell (Kokaku kidotai) (1995)
   206  Akira (1988)
   820  Space Jam (1996)
   797  Timecop (1994)
    62  Stargate (1994)
   152  Sleeper (1973)
   204  Back to the Future (1985)
  1416  No Escape (1994)
   760  Screamers (1995)
    82  Jurassic Park (1993)


## 7 · Langfuse tracing

If Langfuse keys are configured, attach a callback handler to every chain
invocation so cost/latency/inputs/outputs for parse -> retrieve -> rerank ->
explain are all traced under one session. If not configured, this cell is a
no-op -- the chains above already ran fine without it.

In [7]:
LANGFUSE_AVAILABLE = False
langfuse_handler = None

try:
    from langfuse.langchain import CallbackHandler
    if os.environ.get("LANGFUSE_PUBLIC_KEY") and os.environ.get("LANGFUSE_SECRET_KEY"):
        langfuse_handler = CallbackHandler()
        LANGFUSE_AVAILABLE = True
except ImportError:
    pass

print("Langfuse tracing active:", LANGFUSE_AVAILABLE)

if LLM_AVAILABLE and LANGFUSE_AVAILABLE:
    traced = conversational_search(
        "a feel-good comedy for a rainy evening",
    )
    # Re-run one chain call with the handler attached to show how tracing hooks in.
    _ = parse_query_chain.invoke(
        {"query": "a feel-good comedy for a rainy evening"},
        config={"callbacks": [langfuse_handler]},
    )
    print("Traced call sent to Langfuse -- check your project dashboard.")
elif not LANGFUSE_AVAILABLE:
    print("Skipped -- set LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY to enable tracing.")

Langfuse tracing active: False
Skipped -- set LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY to enable tracing.


## Summary

| Chain | Input | Output | Grounding |
|---|---|---|---|
| `parse_query_chain` | free text | JSON search intent | n/a (parsing only) |
| retrieval (`retrieve_candidates`) | search text | real movies from `items.csv` | embeds with the same E5 model as notebook 03 |
| `rerank_chain` | query + candidates | ordered subset of candidate ids | constrained to ids actually retrieved |
| `explain_chain` | one movie's real metadata | 1-2 sentence explanation | only given fields, no invented facts |
| `onboarding_chain` | free-text answers | taste summary -> retrieval | cold start via content vectors, no history needed |

This closes out Phase 1. The two-tower model (04) beats the popularity
baseline and RBM (02), Qdrant (05) makes the collaborative vectors searchable
at scale, and this notebook proves out the RAG + re-rank + explain pattern
that prevents hallucinated titles.

**Next -> Phase 2:** move these chains into `backend/llm_chains.py`, wrap the
whole pipeline (`recommender.py`) behind FastAPI routes (`/chat`,
`/onboarding`, `/explain/{movie_id}`), and put Qdrant/Postgres/Redis/Langfuse
behind `docker-compose.yml`.